In [ ]:
!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 37.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.8/236.8 kB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 66.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.5 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
sentence1 = 'Michael Lutz sucks'
sentence2 = 'Nishkal Hundia is the best'

In [ ]:
combined_sentences = [sentence1, sentence2]

In [ ]:
tokenized_sentences = tokenizer(combined_sentences, padding = True, truncation = True, return_tensors = 'pt')

In [ ]:
tokenized_sentences

{'input_ids': tensor([[  101,  2745, 11320,  5753, 19237,   102,     0,     0,     0,     0,
             0],
        [  101,  9152,  4095, 12902, 15876, 16089,  2050,  2003,  1996,  2190,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
with torch.no_grad():
  model_output = model(**tokenized_sentences) #converts dictionary above into arguments

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

In [ ]:
embedded_sentences = mean_pooling(model_output, tokenized_sentences['attention_mask'])

In [ ]:
len(embedded_sentences[1])

384

In [ ]:
F.cosine_similarity(embedded_sentences[0], embedded_sentences[1], dim=0)

tensor(0.0544)